<div style="background-color:#118AB2; padding: 18px; border-radius: 3px; text-align:center; font-size:2.5em; font-weight:bold; color:#222; margin-bottom:25px; letter-spacing:1px;">
Predicting Donor Response for Social Good 
</div>

# <h2 style="border-bottom: 4px solid #118AB2; width:fit-content; margin: 0 auto 25px auto; padding-bottom:6px; font-weight:bold;">Model Selection</h2>

_Data Mining II 2025/2026_

Project by:
Francisco Gomes (20221810), Margarida Marchão (20221901), Marta Alves (20221890), Pedro Coimbras (20211573)


## <h3 style="border-bottom: 4px solid #118AB2; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">1. Imports and data loading</h3>




In [4]:
"""Importing the libraries needed for data handling, preprocessing, modeling, and evaluation"""
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.tree import DecisionTreeClassifier

"""Adding the project root to the Python path so notebook imports work correctly"""
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)


In [5]:
"""Loading the training dataset for  modeling"""
X_train = pd.read_csv(PROJECT_ROOT / "project_data" / "X_train_clean.csv", index_col=0)
X_val = pd.read_csv(PROJECT_ROOT / "project_data" / "X_val_clean.csv", index_col=0)
X_test = pd.read_csv(PROJECT_ROOT / "project_data" / "X_test_clean.csv", index_col=0)
y_train = pd.read_csv(PROJECT_ROOT / "project_data" / "y_train_clean.csv", index_col=0)
y_val = pd.read_csv(PROJECT_ROOT / "project_data" / "y_val_clean.csv", index_col=0)

X_test.head()

,CHILDREN,DONOR_AGE,FILE_CARD_GIFT,FREQUENCY_STATUS_97NK,INCOME_GROUP,LAST_GIFT_AMT,LIFETIME_CARD_PROM,LIFETIME_GIFT_AMOUNT,LIFETIME_GIFT_COUNT,LIFETIME_MAX_GIFT_AMT,LIFETIME_MIN_GIFT_AMT,LIFETIME_PROM,MEDIAN_HOME_VALUE,MEDIAN_HOUSEHOLD_INCOME,MONTHS_SINCE_FIRST_GIFT,MONTHS_SINCE_LAST_GIFT,MONTHS_SINCE_LAST_PROM_RESP,NUMBER_PROM_12,PCT_ATTRIBUTE1,PCT_ATTRIBUTE2,PCT_ATTRIBUTE3,PCT_ATTRIBUTE4,PCT_OWNER_OCCUPIED,PEP_STAR,PER_CAPITA_INCOME,RECENT_AVG_CARD_GIFT_AMT,RECENT_AVG_GIFT_AMT,RECENT_CARD_RESPONSE_COUNT,RECENT_CARD_RESPONSE_PROP,RECENT_RESPONSE_COUNT,RECENT_RESPONSE_PROP,RECENT_STAR_STATUS,DONOR_GENDER_A,DONOR_GENDER_F,DONOR_GENDER_M,DONOR_GENDER_U,HOME_OWNER_H,HOME_OWNER_U,RECENCY_STATUS_96NK_A,RECENCY_STATUS_96NK_E,RECENCY_STATUS_96NK_F,RECENCY_STATUS_96NK_L,RECENCY_STATUS_96NK_N,RECENCY_STATUS_96NK_S,SES_1,SES_2,SES_3,SES_4,SES_?,URBANICITY_?,URBANICITY_C,URBANICITY_R,URBANICITY_S,URBANICITY_T,URBANICITY_U
CARD_PROM_12,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1.791759,1.098612,1.858672e+31,2.406816,1.0,1.747107,3.846601,3.005947,5.635703,3.074038,4.070850,3.199324,4.029075,7.890031,5.841866,50.0,6.565997e+07,3.332205,2.650016,2.779401,2.146436e+14,4.178163,3.653590,1.252363e+29,0.0,9.447466,0.000000,3.733314,1.307040,0.000000,1.446787,0.068593,2.055337,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1.791759,0.693147,5.052394e+31,3.404478,4.0,1.724603,3.326244,3.668813,5.911490,3.863443,3.679000,3.068795,4.534843,7.986684,5.713733,128.0,1.318816e+09,3.433987,2.411822,2.839627,5.834617e+14,3.789762,4.070957,3.025077e+36,1.0,9.398561,1.704748,3.065606,2.271633,0.435024,2.420318,0.351361,2.175828,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1.945910,1.386294,4.311232e+15,3.049205,2.0,1.747107,3.460522,3.642975,6.073634,3.600842,3.924086,3.199324,4.805732,7.815420,5.841866,130.0,1.484132e+02,3.258097,3.225027,2.715315,1.739275e+18,3.644036,4.213801,1.694889e+28,1.0,9.032529,2.890372,3.490960,1.546545,0.117783,1.658121,0.105261,2.175828,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1.945910,1.098612,3.550938e+25,2.493087,4.0,1.318544,3.460522,2.785341,5.621328,3.119235,3.797886,3.199324,3.896074,7.953855,5.789960,23.0,8.886111e+06,3.367000,2.952525,2.715315,1.318816e+09,3.562349,3.704090,2.038281e+34,1.0,9.393828,2.140066,3.221570,1.739627,0.336472,1.980924,0.310422,2.055337,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1.945910,0.693147,3.550938e+25,2.949613,2.0,1.318544,3.733700,3.642975,5.935571,3.515391,3.981676,3.114224,4.566522,7.864622,5.673323,127.0,8.886111e+06,3.258097,2.650016,2.715315,1.446257e+12,3.955961,4.018418,2.235247e+37,1.0,9.187686,3.044522,3.605972,1.546545,0.117783,1.658121,0.105261,2.175828,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


## <h3 style="border-bottom: 4px solid #118AB2; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">2. Unitary Models</h3>

Rebuilding models from baseline models with proper cleaning and transformations




In [6]:
"""Defining the target and identifier variables and creating the train-validation split"""
TARGET = "TARGET_B"
ID_COLUMN = "CONTROL_NUMBER"
RANDOM_STATE = 5
TRAIN_SIZE = 0.70

In [7]:
"""Defining the first set of baseline classifiers to compare"""
model_configs = {
    "Logistic Regression": {
        "estimator": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "model_family": "linear",
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "model_family": "tree"}}
  

In [8]:


# ── Training & Evaluation ─────────────────────────────────────────────────────
# Assumes the following variables are already defined in the environment:
#   X_train, y_train  — training features and labels
#   X_val,   y_val    — validation features and labels
#   X_test            — test features (no labels)
#   test_ids          — Series/array of CONTROL_NUMBER values for X_test

results = {}     # stores metrics per model
trained_models = {}  # stores fitted estimators

print("=" * 60)
print("MODEL TRAINING & VALIDATION METRICS")
print("=" * 60)

for model_name, config in model_configs.items():
    estimator = config["estimator"]

    # ── Train ────────────────────────────────────────────────────────────────
    estimator.fit(X_train, y_train)
    trained_models[model_name] = estimator

    # ── Predict on train & val ───────────────────────────────────────────────
    train_preds = estimator.predict(X_train)
    val_preds   = estimator.predict(X_val)

    # ── Metrics ──────────────────────────────────────────────────────────────
    train_acc = accuracy_score(y_train, train_preds)
    val_acc   = accuracy_score(y_val,   val_preds)

    train_f1  = f1_score(y_train, train_preds, average="weighted", zero_division=0)
    val_f1    = f1_score(y_val,   val_preds,   average="weighted", zero_division=0)

    results[model_name] = {
        "train_accuracy": train_acc,
        "val_accuracy":   val_acc,
        "train_f1":       train_f1,
        "val_f1":         val_f1,
    }

    # ── Print summary ────────────────────────────────────────────────────────
    print(f"\n{'─' * 60}")
    print(f"  {model_name}  [{config['model_family']}]")
    print(f"{'─' * 60}")
    print(f"  {'Metric':<20} {'Train':>10} {'Validation':>12}")
    print(f"  {'Accuracy':<20} {train_acc:>10.4f} {val_acc:>12.4f}")
    print(f"  {'F1 (weighted)':<20} {train_f1:>10.4f} {val_f1:>12.4f}")
    print(f"\n  Validation Classification Report:")
    print(classification_report(y_val, val_preds, zero_division=0))

# ── Metrics comparison table ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
metrics_df = pd.DataFrame(results).T.rename_axis("Model")
print(metrics_df.to_string())

# ── Test Predictions ──────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("EXPORTING TEST PREDICTIONS")
print("=" * 60)

for model_name, estimator in trained_models.items():
    test_preds = estimator.predict(X_test)

    # Include predicted probabilities for the positive class if available
    if hasattr(estimator, "predict_proba"):
        test_proba = estimator.predict_proba(X_test)[:, 1]
    else:
        test_proba = np.full(len(test_preds), np.nan)

    # Build output DataFrame
    output_df = pd.DataFrame({
        ID_COLUMN:              test_ids.values if hasattr(test_ids, "values") else test_ids,
        f"pred_{TARGET}":       test_preds,
        f"prob_{TARGET}":       test_proba,
    })

    # Safe filename
    safe_name = model_name.lower().replace(" ", "_")
    output_path = f"predictions_{safe_name}.csv"
    output_df.to_csv(output_path, index=False)
    print(f"  ✓ {model_name:25s} → {output_path}  ({len(output_df):,} rows)")

print("\nDone.")

MODEL TRAINING & VALIDATION METRICS


ValueError: Input X contains infinity or a value too large for dtype('float64').

## <h3 style="border-bottom: 4px solid #118AB2; margin: 0 0 25px 0; padding-bottom:6px; font-weight:bold; font-size:1.7em;text-align:left;">X. Exporting Results</h3>

Exporting results.csv


